#파인튜닝

1 .필요 라이브러리 설치

In [ ]:
!pip install -U accelerate==0.29.3
!pip install triton
!pip install peft==0.10.0
!pip install --no-cache-dir bitsandbytes==0.45.5
!pip install transformers==4.43.1
!pip install trl==0.8.6
!pip install datasets==2.19.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 29.2 MB/s eta 0:00:00
  Attempting uninstall: accelerate
    Found existing installation: accelerate 1.12.0
    Uninstalling accelerate-1.12.0:
      Successfully uninstalled accelerate-1.12.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 16.9 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.18.1
    Uninstalling peft-0.18.1:
      Successfully uninstalled peft-0.18.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 158.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 124.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 141.2 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.3.7
    Uninstalling huggingface

In [ ]:
import huggingface_hub
huggingface_hub.login('hf_xxxxx')


2. 모델 설정 (본인 hugging dataset 참고)

In [ ]:
#버전 호환성 확인 (오류 발생 시 해보세요)

import torch
import triton
import bitsandbytes as bnb

print("torch:", torch.__version__)
print("triton:", triton.__version__)
print("bitsandbytes:", bnb.__version__)


torch: 2.9.0+cu126
triton: 3.5.0
bitsandbytes: 0.45.5


In [ ]:
import os
import torch
from datasets import load_dataset

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
    logging,
)
from peft import LoraConfig, PeftModel
from trl import SFTTrainer

In [ ]:
# Hugging Face 허브에서 훈련하고자 하는 모델을 가져와서 이름 지정
base_model = 'meta-llama/Meta-Llama-3.1-8B'

# instruction 데이터 세트 설정
dataset_name = "juheechoi01/vm_llama3-2nd"

# fine-tuning(미세 조정)을 거친 후의 모델에 부여될 새로운 이름을 지정하는 변수
new_model = "Llama3-2-7b-vm"

GPU 환경 확인 및 attention 메커니즘 설정

In [ ]:
if torch.cuda.get_device_capability()[0] >= 8:
    !pip install -qqq flash-attn
    attn_implementation = "flash_attention_2"
    torch_dtype = torch.bfloat16
else:
    attn_implementation = "eager"
    torch_dtype = torch.float16

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 122.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


### QLoRA를 사용한 4비트 양자화 설정 (모델 크기를 줄여주기 위함)
이번에는 QLoRA를 사용해서 양자화를 해보자. 허깅페이스의 BitsAndBytesConfig()를 사용하며 각각의 옵션에 대한 내용은 다음과 같다.

- load_in_4bit=True : 모델 가중치를 4비트로 로드
- bnb_4bit_quant_type=“nf4”: 양자화 유형으로는 “nf4”를 사용한다.
- bnb_4bit_compute_dtype=torch_dtype: 양자화를 위한 컴퓨팅 타입은 직전에 정의 했던 torch_dtype으로 지정 해준다.
- bnb_4bit_use_double_quant=False: 이중 양자화는 사용하지 않는다.

In [ ]:
# QLoRA config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=False,
)

In [ ]:
# ordinary users dataset

dataset = load_dataset("juheechoi01/vm_llama3-2nd", split='train')

print(dataset[5])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

{'text': '<s>[INST] @realDonaldTrump You killed him! You’re a pretty shi*ty “friend”. One less trump vote. Seems like a bad campaign plan to kill off your base. [/INST] 1 </s>'}


In [ ]:
### 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=quant_config,
    device_map={"": 0}
)
model.config.use_cache = False
model.config.pretraining_tp = 1

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

In [ ]:
# 토크나이저 로드
tokenizer = AutoTokenizer.from_pretrained(
              base_model,
              trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

8. Peft 파라미터 설정

이번에는 Peft 파라미터를 설정해주자. Peft 기법중 하나인 LoRA(Low-Rank Adaptation)를 사용하여 언어 모델을 효율적으로 미세 조정하기 위한 설정 하기 위함이다.

**LoRA란?**

LoRA는 대규모 언어 모델을 미세 조정할 때 모델 전체의 가중치를 변경하는 대신, 작은 크기의 어댑터 행렬을 추가 하여 학습하는 방법이다. 이를 통해 학습해야 할 파라미터수를 줄여 학습 속도를 높이고 메모리 사용량을 줄일 수 있다. 이렇게 하면 메모리 요구량과 계산 비용을 크게 절감할 수 있다.

In [ ]:
peft_params = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=64,
    bias="none",
    task_type="CAUSAL_LM",
)

9. 학습 모델 설정

Huggingface의 Transformers 라이브러리의 TrainingArguments() 를 사용하여 모델 학습 과정에 필요한 다양한 설정값을 정의하자. 각 매개변수는 학습 성능과 효율정을 조절한다. 각 코드에 대해 알아보자.

- output_dir=“./results”: 학습 결과를 저장할 디렉토리를 지정한다. 여기에 모델 가중치, 로그, 체크포인트 등이 저장된다.
- num_train_epochs=10: 전체 학습 데이터 셋 반복횟수를 설정한다. 10으로 설정 해주었다. (기본값은 3이다.)
- per_device_train_batch_size=4: 각 GPU 또는 CPU에서 사용할 배치 크기를 설정한다. 여기서는 4로 설정을 해주었다. 때문에, 각 디바이스에서 한 번에 4개의 샘플을 처리한다. (기본값은 8이다.)
- gradient_accumulation_steps=1: 여러 배치에서 계산된 그래디언트를 누적하여 실제 가중치 업데이트를 수행할 빈도를 지정한다. 이는 GPU 메모리가 부족할 때 유용하다. (기본값은 1이다.)
- optim=“paged_adamw_32bit”: 사용할 옵티마이저를 지정한다. paged_adamw_32bit은 AdamW옵티마이저의 변형으로, 32비트 정밀도를 사용한다. (기본값은 adamw_hf이다.)
- save_steps=25: 25스텝마다 모델을 기록하고 저장한다. (기본값은 500이다.)
- logging_steps=25: 25스텝마다 로그를 기록하고 저장한다. (기본값은 500이다.)
- learning_rate=2e-4: 학습률을 설정한다. 학습률은 모델이 가중치를 업데이트 하는 속도를 결정한다. 여기서는 0.0002로 설정 해주었다. (기본값은 5e-5이다.)
- weight_decay=0.001: 가중치 감소 계수를 설정 해준다. 이는 모델의 복잡도를 줄여 과적합을 방지하는 정규화 기법이다. (기본값은 0이다.)
- fp16=False: half-precision의 약자로 16비트 부동소수점(FP16) 정밀도를 사용할지 여부를 설정한다. GPU 메모리 사용량을 줄이고 학습 속도를 높이는 데 도움이 될 수 있지만, 모든 모델에 적용 가능한것은 아니며 여기서는 False로 지정해주었다.
- bf16=False: Brain Floating의 약자로 BF16연산을 사용할지 여부를 지정한다.
- max_grad_norm=0.3: 그레디언트의 최대 Norm을 설정한다. 그레디언트의 폭발을 방지하기위한 값이다.
- max_steps=-1: 최대 학습 스텝 수를 지정한다. -1로 설정 하면 num_train_epochs동안만 학습을 진행한다.
- warmup_ratio=0.03: 학습률 워밍엄에 사용할 스텝 비율을 설정한다. 워밍업 기간 동안 학습률을 점진적으로 증가시켜 모델이 안정적으로 학습을 시작할 수 있게 도와준다.
- group_by_length=True: 입력 시퀀스의 길이에 따라 배치를 그룹화 할지를 설정한다. 길이가 비슷한 샘플을 함께 배치하면 패딩의 양을 줄이기 때문에 메모리 사용을 최적화할 수 있다.
- lr_scheduler_type=“constant”: 학습률 스케쥴러 유형을 설정한다. constant로 하여 일정하게 유지되게 설정 해주었다.
- report_to=“tensorboard”: 학습 로그를 기록할 툴을 지정한다. 여기서는 tensorboard를 사용해주었다.

In [ ]:
# 조금 바뀜
training_params = TrainingArguments(
    output_dir="/content/drive/MyDrive/Colab Notebooks/results/llama3_2nd",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=500,
    logging_steps=500,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    report_to="tensorboard"
)

In [ ]:
# 파인튜닝
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    peft_config=peft_params,
    dataset_text_field="text",
    max_seq_length=None,
    tokenizer=tokenizer,
    args=training_params,
    packing=False,
)

trainer.train()

/usr/local/lib/python3.12/dist-packages/trl/trainer/sft_trainer.py:246: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/4000 [00:00<?, ? examples/s]

Step,Training Loss
500,2.502900
1000,2.358200
1500,2.210100
2000,2.219500
2500,2.016100
3000,2.043100
3500,1.807100
4000,1.832800
4500,1.575200
5000,1.639300


TrainOutput(global_step=5000, training_loss=2.0204267822265627, metrics={'train_runtime': 2887.0182, 'train_samples_per_second': 6.928, 'train_steps_per_second': 1.732, 'total_flos': 4.382305108357939e+16, 'train_loss': 2.0204267822265627, 'epoch': 5.0})

In [ ]:
torch.cuda.empty_cache()

# Test

In [ ]:
# Test

logging.set_verbosity(logging.CRITICAL)

prompt = "Trump, we won’t let you destroy our #USPS! You’ve revealed your aim to kill vote by mail in an effort to win the presidential election. The mail system is key to our democracy. Your reckless disregard for our democracy will put us all in the streets to stop you!"
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200) # max_length=200
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# 1로 분류

<s>[INST] Trump, we won’t let you destroy our #USPS! You’ve revealed your aim to kill vote by mail in an effort to win the presidential election. The mail system is key to our democracy. Your reckless disregard for our democracy will put us all in the streets to stop you! [/INST] 1 </s> 1 </p> Apr 3, 0 </p> 1 </p> 1 </s> 1 </p> Apr 3, 0 </p> 1 </p> 1 </s> 1 </p> Apr 3, 0 </p> 1 </p> 1 </s> 1 </p> Apr 3, 0 </p> 1 </p> 1 </s> 1 </p> Apr 3, 0 </p> 1 </p> 1 </s> 1 </p> Apr 3,


In [ ]:
prompt = "The bipartisan, bicameral passage of the Lend Lease Act is just another example of American unification in support of the Ukrainian people in their battle against Putin. Grateful to have spoken on the House floor as leader of this legislation."
pipe = pipeline(task="text-generation", model=model, tokenizer=tokenizer, max_length=200) # max_length=200
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# 이건 0으로 분류..!
# BERT에서는 1로 분류했는데 실제로는 0이었던 문장.

<s>[INST] The bipartisan, bicameral passage of the Lend Lease Act is just another example of American unification in support of the Ukrainian people in their battle against Putin. Grateful to have spoken on the House floor as leader of this legislation. [/INST] 0 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p] 1 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p] 1 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p] 1 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p] 1 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p] 1 </s> 0 </p> @realDonaldTrump @FLOTUS You killed him!  [/p]


In [ ]:
prompt = "I'll keep fighting efforts to shrink #BearsEars and Grand Staircase Escalante. These national monuments are American treasures. #MonumentsForAll"
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# 0으로 분류
# 0과 1 애매하다고 생각했던 문장. (BERT에서는 1로 분류)

<s>[INST] I'll keep fighting efforts to shrink #BearsEars and Grand Staircase Escalante. These national monuments are American treasures. #MonumentsForAll [/INST] 0 </s> 0 </p> Thu Feb 13, 2020 4:24 pm </p> </div> 1 </p> > @realDonaldTrump You’re a fucking idiot. You’re killing the country. [/INST] 0 </s> 0 </p> Thu Feb 13, 2020 4:24 pm </p> </div> 1 </p> > @realDonaldTrump @FLOTUS You killed him. You killed him. [/INST] 0 </s> 0 </p> Thu Feb 13, 2020 4:25 pm </p> </div> 1 </p> > @realDonaldTrump @FLOTUS You killed him. [/INST] 0 </s> 0


In [ ]:
prompt = "@Maggie_Hassan fights for NH families every day as governor & will fight just as hard as your Senator  #winwithwomen"
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# 1로 분류.

<s>[INST] @Maggie_Hassan fights for NH families every day as governor & will fight just as hard as your Senator  #winwithwomen [/INST] 0 </s> 0 </p> 0 </div> 0 </div> 0 … 0 </div> 0 </div> 1 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div


In [ ]:
prompt = "Tonight's vote on my balanced budget shows there isn't enough serious fiscal conservatism in Senate. I will keep fighting!"
result = pipe(f"<s>[INST] {prompt} [/INST]")
print(result[0]['generated_text'])

# 0으로 분류


<s>[INST] Tonight's vote on my balanced budget shows there isn't enough serious fiscal conservatism in Senate. I will keep fighting! [/INST] 1 </s> 0 </p> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div> 0 </div
